# 01_merge_ic50 — BindingDB + ChEMBL 데이터 합치기

**한 줄 요약:** 두 데이터베이스(BindingDB, ChEMBL)에서 받은 분자와 IC50 값을 **하나의 엑셀 표**로 합치고,
같은 분자가 여러 번 들어온 **중복을 찾아 표시**한다. (이후 노트북들이 쓸 원본 데이터)

**용어:** IC50=저해 세기(작을수록 강함) / SMILES=분자를 나타낸 글자 / canonical SMILES=같은 분자를 같은 글자로 통일한 표준형.
**큰 흐름:** ① 준비/함수 → ② BindingDB 읽기 → ③ ChEMBL 읽기 → ④ 합치기+표준화 → ⑤ 중복 판정 → ⑥ 엑셀 저장

> **📌 이 노트북 읽는 법 (처음이면 여기부터)**
> - **셀** = 코드 한 덩어리. 위에서부터 하나씩 실행(`Shift`+`Enter`). 앞 셀에서 만든 값을 뒤 셀이 쓰니 **순서대로**.
> - 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]** 순서. ③은 그 셀에 **처음 나온** 함수·문법 설명(반복은 생략).
> - `# ...` 은 실행에 영향 없는 **주석**(설명).

### 셀 1 — 준비: 폴더 위치 맞추기
노트북을 어느 폴더에서 열어도 프로젝트 최상위에서 돌도록 위치를 맞춘다(그래야 `data/...` 경로가 맞음).

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

🔎 **코드 뜯어보기 (셀 1)**
- `import os` : **import**=도구 묶음(라이브러리) 가져오기. **os**=폴더·파일 다루는 기본 도구.
- `os.path.isdir('data')` : 괄호 `()`=실행. 'data 폴더가 있나?'를 True/False로 답하는 **함수**.
- `if 조건:` + 들여쓰기 : 조건이 참이면 아래 들여쓴 줄만 실행. 파이썬은 **들여쓰기**로 소속을 표시.
- `os.chdir('..')` : 작업 폴더를 '한 칸 위(`..`)'로 이동. `print(...)`=화면 출력.

### 셀 2 — 도구(라이브러리) 불러오기
이 노트북에 필요한 도구들을 가져온다. `re`(정규식), `numpy`(숫자), `pandas`(표), `rdkit`(분자).

In [ ]:
import re
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

🔎 **코드 뜯어보기 (셀 2)**
- `import re` : **re**=정규식(regular expression) 도구. 글자에서 특정 패턴을 찾는다.
- `import numpy as np` : **numpy**=숫자·배열 계산. **as np**='앞으로 np로 부르겠다'는 별명.
- `import pandas as pd` : **pandas**=엑셀 같은 **표(DataFrame)** 도구. 별명 pd.
- `from rdkit import Chem` : **from A import B**=큰 묶음 A에서 B만 콕 집기. Chem=분자 다루는 부분.
- `RDLogger.DisableLog('rdApp.*')` : RDKit 경고 메시지 끄기(화면 정리).

### 셀 3 — 파일 경로 정하기 + 도우미 함수 2개 만들기
읽고 저장할 파일 경로를 변수에 담고, 뒤에서 재사용할 작은 **함수** 두 개를 정의한다.
- `parse_value`: `'>10000'` 같은 글자에서 **부등호와 숫자**를 분리
- `canon`: SMILES를 RDKit **표준형(canonical)** 으로 변환(같은 분자를 같은 글자로 통일)

In [ ]:
BINDINGDB = "data/bindingdb_hsd17b13.tsv"
CHEMBL = "data/chembl_hsd17b13.tsv"
OUT = "data/HSD17B13_IC50_merged.xlsx"


def parse_value(x):
    """'>10000', '<0.5', '100' → (relation, float)."""
    if pd.isna(x):
        return "", np.nan
    s = str(x).strip()
    m = re.match(r"^\s*([<>=~]+)?\s*([0-9.eE+-]+)", s)
    if not m:
        return "", np.nan
    rel = m.group(1) or "="
    try:
        return rel, float(m.group(2))
    except ValueError:
        return rel, np.nan


def canon(smiles):
    """RDKit canonical SMILES. 실패 시 None."""
    if pd.isna(smiles):
        return None
    mol = Chem.MolFromSmiles(str(smiles))
    return Chem.MolToSmiles(mol) if mol else None

🔎 **코드 뜯어보기 (셀 3)**
- `BINDINGDB = "..."` : `=`로 **문자열**(따옴표로 감싼 글자)을 변수에 저장.
- `def parse_value(x):` : **def**=함수 정의. `x`=입력. 아래 들여쓴 부분이 함수 몸통. 재사용할 작업을 묶는다.
- 함수 첫 줄에서 따옴표 3개로 감싼 글은 **docstring**(함수 설명)이라 실행되지 않는다.
- `if pd.isna(x): return "", np.nan` : `pd.isna`=값이 비었는지. **return**=결과 돌려주기. `np.nan`='숫자 빈 값'.
- `str(x).strip()` : `str(...)`=글자로 바꿈, `.strip()`=양끝 공백 제거.
- `re.match(r"^\s*([<>=~]+)?\s*([0-9.eE+-]+)", s)` : **정규식**으로 s 앞부분을 검사. `r"..."`=백슬래시를 그대로 쓰는 문자열. 대략 '(부등호 같은 기호)(숫자)' 모양을 찾음. 결과가 없으면(`if not m`) 실패 처리.
- `m.group(1)`, `m.group(2)` : 정규식에서 괄호 `( )`로 묶은 **1번째·2번째 조각**(여기선 부등호, 숫자)을 꺼냄. `A or "="`=A가 비면 '='로.
- `try: ... except ValueError: ...` : 시도해보고 숫자 변환 오류가 나면 대체 처리.
- `Chem.MolFromSmiles(...)` : SMILES 글자 → 분자. `Chem.MolToSmiles(mol)` : 분자 → **표준 SMILES 글자**. `A if 조건 else B`=조건부 값.

### 셀 4 — BindingDB 파일 읽고 정리
BindingDB TSV에서 필요한 열만 읽어, IC50 값이 있는 행만 남기고, 부등호/숫자를 분리해 **표준 형식(bdb_out)** 으로 만든다.

In [ ]:
bdb = pd.read_csv(BINDINGDB, sep="\t", engine="python", on_bad_lines="skip",
                  usecols=["Ligand SMILES", "IC50 (nM)",
                           "BindingDB Ligand Name", "ChEMBL ID of Ligand"])
bdb = bdb[bdb["IC50 (nM)"].notna()].copy()
rel_val = bdb["IC50 (nM)"].apply(parse_value)
bdb["relation"] = [r for r, _ in rel_val]
bdb["ic50_nM"] = [v for _, v in rel_val]
bdb_out = pd.DataFrame({
    "source": "BindingDB",
    "compound_id": bdb["ChEMBL ID of Ligand"].fillna(bdb["BindingDB Ligand Name"]),
    "smiles": bdb["Ligand SMILES"],
    "ic50_nM": bdb["ic50_nM"],
    "relation": bdb["relation"],
    "pChEMBL": np.nan,
    "std_type": "IC50",
})

🔎 **코드 뜯어보기 (셀 4)**
- `pd.read_csv(경로, sep="\t", ...)` : 표 파일 읽기. `sep="\t"`=탭으로 열 구분(TSV), `usecols=[...]`=**필요한 열만** 읽기, `on_bad_lines="skip"`=깨진 줄은 건너뜀.
- `bdb["IC50 (nM)"].notna()` : 그 열이 비어있지 **않은지** True/False. `bdb[조건]`=참인 행만 고르기(**불리언 인덱싱**). `.copy()`=복사(원본 보호).
- `.apply(parse_value)` : 각 값에 우리가 만든 `parse_value` 함수를 **일괄 적용**.
- `[r for r, _ in rel_val]` : 리스트 컴프리헨션 — 각 (부등호, 숫자) 쌍에서 부등호 r만 뽑아 리스트로. `_`=안 쓸 값.
- `pd.DataFrame({ "열이름": 값, ... })` : **딕셔너리**로 표 만들기(키=열 이름). `.fillna(다른값)`=빈 값을 다른 값으로 채움.

### 셀 5 — ChEMBL 파일 읽고 IC50만 남기기
ChEMBL TSV를 읽어 측정 종류가 **IC50인 행만** 고른다.

In [ ]:
chembl = pd.read_csv(CHEMBL, sep="\t", engine="python", on_bad_lines="skip",
                     usecols=["Molecule ChEMBL ID", "Smiles", "Standard Type",
                              "Standard Relation", "Standard Value",
                              "Standard Units", "pChEMBL Value"])
chembl = chembl[chembl["Standard Type"].astype(str).str.upper() == "IC50"].copy()

🔎 **코드 뜯어보기 (셀 5)**
- `.astype(str)` : 열을 글자형으로 변환. `.str.upper()` : 글자를 대문자로(비교를 통일하려고).
- `chembl[조건].copy()` : IC50인 행만 골라 복사. `== "IC50"`=값이 IC50과 같은지 비교.

### 셀 6 — ChEMBL 단위를 nM으로 통일
IC50 값의 단위가 uM·M이면 nM으로 환산해 맞추고, ChEMBL도 **표준 형식(chembl_out)** 으로 만든다.

In [ ]:
val = pd.to_numeric(chembl["Standard Value"], errors="coerce")
units = chembl["Standard Units"].astype(str).str.strip()
val_nM = np.where(units == "uM", val * 1000, val)
val_nM = np.where(units == "M", val * 1e9, val_nM)
chembl["ic50_nM"] = np.where(units.isin(["nM", "uM", "M"]), val_nM, np.nan)
chembl_out = pd.DataFrame({
    "source": "ChEMBL",
    "compound_id": chembl["Molecule ChEMBL ID"],
    "smiles": chembl["Smiles"],
    "ic50_nM": chembl["ic50_nM"],
    "relation": chembl["Standard Relation"].astype(str).str.replace("'", "").fillna("="),
    "pChEMBL": pd.to_numeric(chembl["pChEMBL Value"], errors="coerce"),
    "std_type": "IC50",
})

🔎 **코드 뜯어보기 (셀 6)**
- `pd.to_numeric(값, errors="coerce")` : 글자를 **숫자로** 변환. 실패하면 빈 값(NaN)으로.
- `np.where(조건, A, B)` : 조건이 참인 자리는 A, 아니면 B로 채우는 **일괄 선택**. 여기선 단위별로 환산값 적용.
- `.isin(["nM","uM","M"])` : 값이 이 목록 안에 있는지 True/False.
- `.str.replace("'", "")` : 글자에서 작은따옴표 제거.

### 셀 7 — 두 데이터 합치고 canonical SMILES 만들기
BindingDB와 ChEMBL을 위아래로 이어 붙이고, 각 분자의 표준 SMILES를 계산한다(중복 판정의 기준).

In [ ]:
df = pd.concat([bdb_out, chembl_out], ignore_index=True)
df = df[df["smiles"].notna() & (df["smiles"].astype(str).str.len() > 0)].copy()
df["canonical_smiles"] = df["smiles"].apply(canon)
df["valid_structure"] = df["canonical_smiles"].notna()

🔎 **코드 뜯어보기 (셀 7)**
- `pd.concat([A, B], ignore_index=True)` : 표 A와 B를 **위아래로**(행 방향) 이어붙임. `ignore_index=True`=행 번호 새로 매김.
- `df["smiles"].notna() & (... .str.len() > 0)` : 두 조건을 **`&`(그리고)** 로 결합 — 값이 있고 길이가 0보다 큰 행.
- `df["canonical_smiles"] = df["smiles"].apply(canon)` : 각 SMILES에 `canon` 적용해 **새 열** 추가.

### 셀 8 — 중복 판정 (같은 표준 SMILES = 같은 물질)
표준 SMILES가 같은 행을 **중복**으로 표시하고, 몇 번 나왔는지(dup_count)도 센다.

In [ ]:
key = df["canonical_smiles"].fillna(
    pd.Series("INVALID_" + df.index.astype(str), index=df.index))
counts = key.map(key.value_counts())
df["dup_count"] = counts.values
df["is_duplicate"] = df["dup_count"] > 1

🔎 **코드 뜯어보기 (셀 8)**
- `.fillna(pd.Series(...))` : 표준 SMILES가 없는(무효) 행은 임시 고유값으로 채워 서로 다른 것으로 취급.
- `key.value_counts()` : 각 값이 **몇 번 나오는지** 셈. `key.map(...)` : 각 행을 그 개수로 바꿔 붙임(=이 물질이 총 몇 번?).
- `df["dup_count"] > 1` : 2번 이상이면 True → 중복.

### 셀 9 — 중복 그룹 번호 매기기
같은 물질끼리 같은 그룹 번호를 붙인다(엑셀에서 묶어 보기 위함).

In [ ]:
dup_keys = sorted(k for k, c in key.value_counts().items()
                  if c > 1 and not str(k).startswith("INVALID_"))
gid = {k: i + 1 for i, k in enumerate(dup_keys)}
df["dup_group"] = key.map(gid).astype("Int64")

🔎 **코드 뜯어보기 (셀 9)**
- `sorted(...)` : 정렬된 목록 만들기. `for k, c in key.value_counts().items()` : (값 k, 개수 c)를 함께 반복.
- `{k: i + 1 for i, k in enumerate(dup_keys)}` : **딕셔너리 컴프리헨션** — 각 물질에 1,2,3… 그룹 번호 부여.
- `.astype("Int64")` : 정수형(빈 값 허용)으로 변환.

### 셀 10 — 소스 간 중복 표시
같은 물질이 **BindingDB와 ChEMBL 양쪽에** 있으면 별도로 표시한다.

In [ ]:
src_per_key = df.groupby(key)["source"].transform(lambda s: s.nunique())
df["cross_source_dup"] = (src_per_key > 1) & df["is_duplicate"]

🔎 **코드 뜯어보기 (셀 10)**
- `df.groupby(key)["source"]` : 같은 물질(key)끼리 **묶기**, 그 안의 source 열을 봄.
- `.transform(lambda s: s.nunique())` : **lambda**=이름 없는 즉석 함수. `s.nunique()`=서로 다른 값 개수. 각 그룹의 소스 종류 수를 원래 행에 되돌려 붙임.
- `(A) & df["is_duplicate"]` : 소스가 2종류 이상이면서 중복인 것.

### 셀 11 — 보기 좋게 정렬 + 열 순서 정리
중복 그룹이 위로 모이도록 정렬하고, 열 순서를 지정한다.

In [ ]:
df = df.sort_values(["is_duplicate", "dup_group", "canonical_smiles", "source"],
                    ascending=[False, True, True, True]).reset_index(drop=True)

col_order = ["source", "compound_id", "smiles", "canonical_smiles", "ic50_nM",
             "relation", "pChEMBL", "std_type", "valid_structure",
             "is_duplicate", "dup_group", "dup_count", "cross_source_dup"]
df = df[col_order]

🔎 **코드 뜯어보기 (셀 11)**
- `df.sort_values([열들], ascending=[...])` : 지정한 여러 열 기준으로 **정렬**. ascending=True(오름차순)/False(내림차순)를 열마다 지정.
- `.reset_index(drop=True)` : 정렬 후 행 번호를 0,1,2…로 새로 매김.
- `df = df[col_order]` : `col_order`(열 이름 리스트) 순서대로 **열을 재배치**.

### 셀 12 — Excel로 저장 (전체 + 중복만 두 시트)
정리된 표를 엑셀 파일로 저장한다. 시트 두 개: 전체(all_data)와 중복만(duplicates_only).

In [ ]:
dups = df[df["is_duplicate"]].copy()
with pd.ExcelWriter(OUT, engine="openpyxl") as w:
    df.to_excel(w, sheet_name="all_data", index=False)
    dups.to_excel(w, sheet_name="duplicates_only", index=False)

🔎 **코드 뜯어보기 (셀 12)**
- `with pd.ExcelWriter(경로, engine="openpyxl") as w:` : **with**=파일을 열고 블록이 끝나면 자동으로 닫는 구문. `w`=엑셀 작성기.
- `df.to_excel(w, sheet_name="all_data", index=False)` : 표를 지정 **시트 이름**으로 저장(index=False=행번호 제외).

### 셀 13 — 중복 행을 노란색으로 칠하기
엑셀을 다시 열어 **중복인 행 전체를 노란색**으로 강조한다(눈으로 찾기 쉽게).

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
wb = load_workbook(OUT)
fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
ws = wb["all_data"]
dup_col = col_order.index("is_duplicate") + 1
for row in range(2, ws.max_row + 1):
    if ws.cell(row=row, column=dup_col).value is True:
        for c in range(1, len(col_order) + 1):
            ws.cell(row=row, column=c).fill = fill
wb.save(OUT)

🔎 **코드 뜯어보기 (셀 13)**
- `from openpyxl import load_workbook` : 엑셀을 세밀히 편집하는 라이브러리. `load_workbook(OUT)`=저장한 엑셀 다시 열기.
- `PatternFill(start_color="FFFF00", ...)` : 채우기 색 정의(FFFF00=노랑).
- `for row in range(2, ws.max_row + 1):` : 2번째 행부터 끝까지 반복(1행은 제목). `ws.cell(row, column).value` : 특정 칸 값 읽기.
- `ws.cell(...).fill = fill` : 그 칸에 색 칠하기. `wb.save(OUT)` : 저장.

### 셀 14 — 요약 출력
총 몇 행인지, 중복이 몇 개인지 등 결과 요약을 화면에 출력한다.

In [ ]:
print("저장 완료 →", OUT)
print(f"총 행: {len(df)} (BindingDB {int((df['source']=='BindingDB').sum())}, "
      f"ChEMBL {int((df['source']=='ChEMBL').sum())})")
print(f"파싱 실패(무효 구조): {int((~df['valid_structure']).sum())}")
print(f"중복 물질 그룹 수: {df['dup_group'].dropna().nunique()}")
print(f"중복에 속한 행 수: {int(df['is_duplicate'].sum())}")
print(f"소스 간(BindingDB↔ChEMBL) 중복 행 수: {int(df['cross_source_dup'].sum())}")
print(f"고유 물질 수(canonical 기준): {df['canonical_smiles'].nunique()}")

🔎 **코드 뜯어보기 (셀 14)**
- `f"총 행: {len(df)} ..."` : **f-문자열** — 앞에 f를 붙이면 `{ }` 안의 값/식을 글자에 바로 끼워 넣는다.
- `(df['source']=='BindingDB').sum()` : 조건이 참(True=1)인 개수를 합해 **개수 세기**.
- `.nunique()` : 서로 다른 값의 개수. `~df['valid_structure']` : `~`=부정(True↔False).